# Lezione 13 - Memoria dell'Agente con Cognee Knowledge Graphs


## Configurazione

Questo notebook dimostra come costruire un **assistente di programmazione** intelligente con memoria persistente utilizzando i grafi di conoscenza di [**Cognee**](https://www.cognee.ai/) e il **Microsoft Agent Framework** (MAF).

Cognee trasforma testo non strutturato in un grafo di conoscenza strutturato e interrogabile supportato da vettori di embedding — offrendo al tuo agente una memoria a lungo termine ricca e consapevole delle relazioni.

### Cosa Imparerai
1. **Costruire Grafi di Conoscenza**: Trasformare profili di sviluppatori e best practice in conoscenza strutturata e interrogabile.
2. **Integrare Cognee con MAF**: Usare funzioni `@tool` per permettere a un agente MAF di interrogare il grafo di conoscenza di Cognee.
3. **Conversazioni Consapevoli della Sessione**: Mantenere il contesto attraverso più domande nella stessa sessione.
4. **Memoria a Lungo Termine**: Conservare conoscenze importanti attraverso le sessioni e recuperarle in nuove conversazioni.

### Prerequisiti
- Python 3.9+
- Redis in esecuzione localmente (`docker run -d -p 6379:6379 redis`) per la gestione delle sessioni
- Una chiave API LLM (es. OpenAI) — impostare `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (richiesto per le sessioni Cognee)
- Un progetto Microsoft Foundry con un modello chat distribuito
- `AZURE_AI_PROJECT_ENDPOINT` e `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI autenticato (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Tipi di Memoria dell'Agente

Questo notebook esplora gli stessi tre tipi di memoria del notebook principale della Lezione 13, ma usa Cognee come backend per la memoria a lungo termine:

| Tipo di Memoria | Meccanismo | Durata |
|---|---|---|
| **Operativa** | `agent.create_session()` (MAF) | Singola conversazione |
| **A Breve Termine** | Cache sessione Cognee (Redis) | Singola sessione |
| **A Lungo Termine** | Grafo di conoscenza Cognee + vettori | Permanente |

### Architettura della Memoria di Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Preparare lo storage Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Parte 1 — Costruire la Base di Conoscenza

Inseriamo tre tipi di dati per creare una base di conoscenza completa per il nostro assistente di codifica:

1. **Profilo dello Sviluppatore** — competenze personali e background tecnico
2. **Best Practice di Python** — lo Zen di Python con linee guida pratiche
3. **Conversazioni Storiche** — sessioni di Q&A passate tra sviluppatori e assistenti AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualizza il Grafo della Conoscenza

Cognee può generare una visualizzazione HTML interattiva delle entità e delle relazioni che ha estratto.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Arricchisci la memoria con Memify

`memify()` analizza il knowledge graph e genera regole intelligenti — identificando schemi, migliori pratiche e relazioni tra concetti.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Parte 2 — Agente MAF con Cognee Tools

Ora creiamo un agente MAF che può interrogare il grafo della conoscenza di Cognee tramite le funzioni `@tool`. Questo permette all'agente di sfruttare la piena potenza della ricerca semantica consapevole del grafo mantenendo il contesto conversazionale attraverso le sessioni.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Memoria di Lavoro con le Sessioni

La `AgentSession` (creata tramite `agent.create_session()`) fornisce memoria di lavoro all'interno di una sessione. L'agente può fare riferimento a messaggi precedenti mentre interroga anche il grafo della conoscenza a lungo termine di Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nuova sessione — La memoria a lungo termine persiste

Iniziare una nuova sessione cancella la memoria di lavoro, ma il grafo di conoscenza Cognee è ancora disponibile. L'agente può recuperare la stessa conoscenza a lungo termine in una conversazione completamente nuova.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sommario

In questo notebook hai costruito un assistente di codifica che combina la **memoria di lavoro di MAF** (`agent.create_session()`) con il **grafo di conoscenza a lungo termine di Cognee**.

### Cosa hai imparato
1. **Costruzione del grafo di conoscenza**: Cognee acquisisce testo non strutturato e costruisce un grafo + memoria vettoriale.
2. **Arricchimento del grafo con memify**: Fatti derivati e relazioni più ricche sopra il tuo grafo esistente.
3. **Integrazione MAF + Cognee**: le funzioni `@tool` permettono a un agente MAF di interrogare naturalmente il grafo di Cognee.
4. **Memoria di lavoro + memoria a lungo termine**: `AgentSession` (via `agent.create_session()`) fornisce il contesto della sessione mentre Cognee fornisce conoscenza persistente.
5. **Ricerca filtrata con NodeSets**: indirizza specifici sottoinsiemi del grafo di conoscenza (ad es. solo principi).

### Punti chiave
- **Cognee** trasforma il testo grezzo in una memoria strutturata e consapevole delle relazioni — più potente di un archivio vettoriale piatto.
- Le **funzioni `@tool`** collegano gli agenti MAF e i sistemi di conoscenza esterni in modo pulito.
- **`AgentSession`** (tramite `agent.create_session()`) mantiene separato il contesto per conversazione dalla conoscenza a lungo termine.
- Lo stesso grafo di conoscenza serve più sessioni e agenti.

### Applicazioni nel mondo reale
- **Copiloti per sviluppatori**: Revisione del codice, analisi degli incidenti, assistenti di architettura
- **Copiloti rivolti ai clienti**: Agenti di supporto su documentazione prodotto, FAQ e note CRM
- **Copiloti esperti interni**: Assistenti per policy, legali o sicurezza che ragionano su linee guida
- **Layer dati unificati**: Combina dati strutturati e non strutturati in un unico grafo interrogabile

### Prossimi passi
- Sperimenta la consapevolezza temporale in Cognee
- Definisci un'ontologia OWL per la qualità del grafo specifica del dominio
- Aggiungi feedback degli utenti per migliorare il recupero nel tempo
- Scala a sistemi multi-agente che condividono lo stesso layer di memoria Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
